# Lesson 9.2: How Do You RAG Over Structured Data Without Writing SQL by Hand?

**Companion notebook for Lesson 9.2**

---

| Section | What you will build |
|---|---|
| 1. Two Kinds of Knowledge | Prove vector search fails on numeric questions — with a live SQLite DB |
| 2. Query Router | Rule-based, embedding-based, and LLM-based routers side by side |
| 3. Text-to-SQL Basics | Vanilla → few-shot → CoT → self-debugging loop |
| 4. Schema Linking | Retrieve relevant tables before generating SQL (RAG-inside-RAG) |
| 5. Hybrid Pipeline | Full `hybrid_rag()` wiring vector search + Text-to-SQL + synthesis |
| 6. Production Concerns | Read-only access, LIMIT injection, PII filtering, caching |
| 7. Claude API | Real Text-to-SQL with Claude and execution refinement |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Built-in (no install):** `sqlite3`  
**Optional (Section 7):** `anthropic`

> **Database used throughout:** an in-memory SQLite database with five tables — `customers`,
> `orders`, `products`, `order_items`, `regions` — plus four plaintext policy documents.
> That split (structured facts vs. narrative text) is the hybrid RAG problem in miniature.


In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional — Section 7

%matplotlib inline
import os, re, sqlite3, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime, timedelta
import random

warnings.filterwarnings('ignore')
os.environ['OMP_NUM_THREADS']         = '1'
os.environ['MKL_NUM_THREADS']         = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3
random.seed(42)
np.random.seed(42)

def show_plot():
    plt.tight_layout()
    plt.show()

print('Imports ready.')


In [ ]:
# ── Build in-memory SQLite database ─────────────────────────────────────────
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE regions (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    launch_year INTEGER
);

CREATE TABLE customers (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    email       TEXT,
    region      TEXT,
    signup_date TEXT,
    tier        TEXT
);

CREATE TABLE products (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    category    TEXT,
    unit_price  REAL
);

CREATE TABLE orders (
    id            INTEGER PRIMARY KEY,
    customer_id   INTEGER REFERENCES customers(id),
    order_date    TEXT,
    status        TEXT,
    total_amount  REAL
);

CREATE TABLE order_items (
    id          INTEGER PRIMARY KEY,
    order_id    INTEGER REFERENCES orders(id),
    product_id  INTEGER REFERENCES products(id),
    quantity    INTEGER,
    unit_price  REAL
);
""")

# ── Seed data ──────────────────────────────────────────────────────────────
REGIONS = [(1,'North America',2018),(2,'Europe',2019),(3,'APAC',2021),
           (4,'LATAM',2022),(5,'MEA',2023)]
conn.executemany('INSERT INTO regions VALUES (?,?,?)', REGIONS)

PRODUCT_DATA = [
    ('DataSync Pro', 'Software', 299.0),
    ('Analytics Suite', 'Software', 499.0),
    ('StoragePack 1TB', 'Hardware', 149.0),
    ('API Credits (1k)', 'Service', 49.0),
    ('Support Plus', 'Service', 199.0),
    ('CloudBackup', 'Software', 99.0),
]
conn.executemany('INSERT INTO products (name,category,unit_price) VALUES (?,?,?)', PRODUCT_DATA)

NAMES  = ['Alice Kim','Bob Chen','Carol Diaz','Dan Patel','Eve Muller',
          'Frank Osei','Grace Liu','Hana Berg','Ivan Soto','Jess Park',
          'Kai Tanaka','Leila Nour','Marco Bianchi','Nadia Volkov','Omar Hassan']
TIERS  = ['free','free','pro','pro','pro','enterprise','enterprise']
REGION_NAMES = [r[1] for r in REGIONS]

base_date = datetime(2023, 1, 1)
for i, name in enumerate(NAMES):
    days = random.randint(0, 700)
    signup = (base_date + timedelta(days=days)).strftime('%Y-%m-%d')
    conn.execute(
        'INSERT INTO customers (name,email,region,signup_date,tier) VALUES (?,?,?,?,?)',
        (name, name.lower().replace(' ','.')+"@techco.com",
         random.choice(REGION_NAMES), signup, random.choice(TIERS))
    )

STATUSES = ['shipped','shipped','shipped','pending','refunded','cancelled']
order_id = 1
for cust_id in range(1, 16):
    n_orders = random.randint(1, 8)
    for _ in range(n_orders):
        days_ago = random.randint(1, 400)
        odate    = (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d')
        status   = random.choice(STATUSES)
        amount   = round(random.uniform(49, 1200), 2)
        conn.execute(
            'INSERT INTO orders (id,customer_id,order_date,status,total_amount) VALUES (?,?,?,?,?)',
            (order_id, cust_id, odate, status, amount)
        )
        order_id += 1

conn.commit()

def qry(sql, params=()):
    """Execute SQL and return list-of-dicts."""
    cur = conn.execute(sql, params)
    cols = [d[0] for d in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]

print('Database ready.')
stats = qry('SELECT (SELECT COUNT(*) FROM customers) c, '
             '(SELECT COUNT(*) FROM orders) o, '
             '(SELECT COUNT(*) FROM products) p')
print(f"  customers: {stats[0]['c']}  orders: {stats[0]['o']}  products: {stats[0]['p']}")


In [ ]:
# ── Policy / narrative documents (the unstructured side) ─────────────────────
TEXT_DOCS = [
    {'id': 'policy_returns', 'title': 'Returns & Refunds Policy',
     'text': (
         'Customers may return any product within 30 days of purchase for a full refund. '
         'Electronics and hardware must be returned in original packaging with no physical damage. '
         'Refunds are processed within 5-7 business days after the item is received. '
         'To initiate a return, email returns@techco.com or use the self-service portal. '
         'Software licenses are non-refundable once activated, except when the product is defective.'
     )},
    {'id': 'memo_regions', 'title': 'Regional Expansion Memo (2023)',
     'text': (
         'TechCo launched in North America in 2018 and Europe in 2019. '
         'APAC expansion followed in 2021, showing the fastest adoption curve of any region. '
         'LATAM launched in 2022 and MEA in 2023, both in early growth phases. '
         'LATAM and MEA carry higher customer acquisition costs but improving unit economics. '
         'Regional managers should expect 18-24 months to breakeven in new markets.'
     )},
    {'id': 'policy_tiers', 'title': 'Customer Tier Benefits',
     'text': (
         'TechCo offers three customer tiers. '
         'Free tier includes core product features with 2GB storage and community support. '
         'Pro tier at $29/month adds unlimited storage, priority email support, and API access. '
         'Enterprise tier adds dedicated SLA guarantees, custom integrations, and a named account manager. '
         'Upgrades take effect immediately; downgrades take effect at the next billing cycle.'
     )},
    {'id': 'memo_shipping', 'title': 'Shipping & Fulfillment Policy',
     'text': (
         'Standard US shipping costs $9.99 and takes 5-7 business days from our Ohio warehouse. '
         'Expedited 2-day shipping costs $24.99. '
         'Orders over $150 qualify for free standard shipping. '
         'International orders: duties and taxes are the buyer responsibility. '
         'All hardware orders include a tracking number emailed upon dispatch.'
     )},
]

print(f'Text corpus: {len(TEXT_DOCS)} policy documents')
for d in TEXT_DOCS:
    print(f"  [{d['id']}] {d['title']}")


---
## 1. Two Kinds of Knowledge, Two Kinds of Retrieval

**The failure mode to avoid:**  
A text chunk might say *"returns averaged 4-6% of gross sales last year."*  
Vector RAG finds that chunk. The LLM uses that stale percentage to answer
*"how many returns did we process last month?"*  
The actual answer is in your database — and it's different.

This isn't a retrieval quality problem. It's a **tool mismatch**: you sent a counting
question to a search engine.


In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

doc_texts  = [d['text'] for d in TEXT_DOCS]
doc_embeds = embedder.encode(doc_texts, convert_to_tensor=True, show_progress_bar=False)

def vector_retrieve(query, top_k=2):
    q_emb  = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, doc_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(TEXT_DOCS[i], float(scores[i])) for i in idx]


# The real answer from the database
real_refunds = qry("""
    SELECT COUNT(*) as count, ROUND(SUM(total_amount),2) as total
    FROM orders
    WHERE status = 'refunded'
      AND order_date >= date('now', '-30 days')
""")
real_total   = qry("SELECT ROUND(SUM(total_amount),2) as revenue FROM orders WHERE status='shipped'")
real_pct     = round(real_refunds[0]['total'] / real_total[0]['revenue'] * 100, 1) if real_total[0]['revenue'] else 0

print('=== Vector-Only RAG vs. SQL on a counting question ===\n')
q = 'How many orders were refunded last month and what percentage of revenue is that?'
print(f'Question: {q}\n')

top_doc, score = vector_retrieve(q)[0]
print(f'Top text match (score={score:.3f}): [{top_doc["id"]}] "{top_doc["title"]}"')
print(f'  Text says: "{top_doc["text"][:120]}..."')
print()
print('Vector RAG answer: "Returns averaged 4-6% of gross sales last year" (stale, from policy doc)')
print()
print('SQL answer (actual database):')
print(f'  Refunded orders last 30 days : {real_refunds[0]["count"]}')
print(f'  Refund value                 : ${real_refunds[0]["total"]}')
print(f'  % of shipped revenue         : {real_pct}%')
print()
print('The text gave you last year. The DB gives you right now.')
print('This is not a retrieval quality problem — it is a tool mismatch.')


---
## 2. The Query Router

The router reads the question and decides: *search the docs* (vector), *query the DB* (sql),
or *do both* (hybrid).

Three implementations, from cheapest to most flexible:

| Router type | Speed | Cost | Best when |
|---|---|---|---|
| Rule-based (regex) | ~0 ms | free | Narrow domain, predictable phrasing |
| Embedding-based | ~10 ms | free | Grows with categories, handles paraphrasing |
| LLM-based | ~300 ms | token cost | Open-ended, ambiguous, evolving schema |


In [ ]:
# ── Router 1: Rule-based (regex keyword matching) ────────────────────────────

SQL_SIGNALS = [
    r'how many', r'how much', r'count', r'total', r'sum', r'average', r'avg',
    r'top \d+', r'list all', r'show me.*who', r'customers? (who|that)',
    r'last \d+ days?', r'last (week|month|year)', r'revenue', r'orders?',
    r'rank', r'highest', r'lowest', r'most', r'least',
]

VECTOR_SIGNALS = [
    r'policy', r'how does', r'what is', r'explain', r'why', r'difference between',
    r'what are the', r'describe', r'when (should|do|can)',
]

def rule_router(question):
    q   = question.lower()
    sql = any(re.search(p, q) for p in SQL_SIGNALS)
    vec = any(re.search(p, q) for p in VECTOR_SIGNALS)
    if sql and vec:
        return 'hybrid'
    if sql:
        return 'sql'
    return 'vector'


ROUTER_TEST = [
    ("What's our return policy for electronics?",          'vector'),
    ("How many returns did we process last month?",        'sql'),
    ("Why did returns spike in March, and by how much?",   'hybrid'),
    ("Explain the difference between gross and net revenue.", 'vector'),
    ("Show me customers who haven't ordered in 90 days.",  'sql'),
    ("What are the top 5 products by revenue?",            'sql'),
    ("What is our shipping policy?",                       'vector'),
    ("How many pro-tier customers do we have in APAC?",    'sql'),
]

print(f'{"Question":<55} {"Expected":<10} {"Rule Router"}')
print('-' * 85)
correct = 0
for q, expected in ROUTER_TEST:
    got  = rule_router(q)
    mark = 'OK' if got == expected else 'MISS'
    if got == expected:
        correct += 1
    print(f'{q:<55} {expected:<10} {got:<10} {mark}')
print(f'\nRule router accuracy: {correct}/{len(ROUTER_TEST)}')


In [ ]:
# ── Router 2: Embedding-based ────────────────────────────────────────────────
# Classify by similarity to a few labelled example questions per category.
# Handles paraphrasing better than regex. Scales to many categories.

ANCHOR_QUESTIONS = {
    'sql': [
        'How many customers signed up last week?',
        'What is the total revenue this quarter?',
        'List the top 5 customers by order count.',
        'How many orders are still pending?',
        'What is the average order value?',
    ],
    'vector': [
        'What is our refund policy?',
        'How does the pro tier differ from free?',
        'Explain our shipping process.',
        'What are the benefits of enterprise tier?',
        'Why did we expand to APAC first?',
    ],
    'hybrid': [
        'Why are refunds up and by how much?',
        'How does our return rate compare to policy targets?',
        'Explain the revenue drop and show the numbers.',
        'Which regions launched last and what are their order counts?',
    ],
}

# Embed all anchor questions, keeping track of their category label
anchor_texts, anchor_labels = [], []
for label, questions in ANCHOR_QUESTIONS.items():
    for q in questions:
        anchor_texts.append(q)
        anchor_labels.append(label)

anchor_embeds = embedder.encode(anchor_texts, convert_to_tensor=True, show_progress_bar=False)

def embed_router(question):
    q_emb  = embedder.encode(question, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, anchor_embeds)[0].cpu().numpy()
    top_k  = np.argsort(scores)[::-1][:5]  # look at top-5 neighbours
    # majority vote among the top-5 nearest anchors
    votes  = {}
    for i in top_k:
        lbl = anchor_labels[i]
        votes[lbl] = votes.get(lbl, 0) + 1
    return max(votes, key=votes.get)


print(f'{"Question":<55} {"Expected":<10} {"Embed Router"}')
print('-' * 85)
correct = 0
for q, expected in ROUTER_TEST:
    got  = embed_router(q)
    mark = 'OK' if got == expected else 'MISS'
    if got == expected:
        correct += 1
    print(f'{q:<55} {expected:<10} {got:<10} {mark}')
print(f'\nEmbedding router accuracy: {correct}/{len(ROUTER_TEST)}')
print()
print('Embedding router handles paraphrasing well because semantic similarity')
print('captures intent, not just keyword presence.')


In [ ]:
# ── Router 3: LLM-based (mock — real version in Section 7) ──────────────────

ROUTER_PROMPT = """\
Classify the user question into ONE category:

- sql    : Requires counting, aggregating, filtering, or looking up specific
           records in a database. Keywords: how many, total, average, top N, list all...
- vector : Asks about concepts, policies, explanations, or qualitative information.
           Keywords: explain, policy, how does, why, difference between...
- hybrid : Needs BOTH factual numbers from the DB AND context/explanation from docs.

Question: {question}
Category (respond with exactly one word: sql, vector, or hybrid):
"""

class MockRouterLLM:
    """
    Simulates an LLM-based router.
    Falls back to embedding router with added heuristics for hybrid detection.
    In production: replace the classify() call with a real LLM API call.
    """

    HYBRID_SIGNALS = ['why.*and', 'explain.*how many', 'compare.*numbers',
                      'context.*revenue', 'reason.*count', 'spike', 'drop']

    def classify(self, question):
        q = question.lower()
        if any(re.search(p, q) for p in self.HYBRID_SIGNALS):
            return 'hybrid'
        return embed_router(question)   # delegate to embedding router for others


llm_router = MockRouterLLM()

print('=== Router comparison across all three implementations ===\n')
print(f'{"Question":<52} {"Expected":<9} {"Rule":<9} {"Embed":<9} {"LLM"}')
print('-' * 95)
for q, expected in ROUTER_TEST:
    rule  = rule_router(q)
    emb   = embed_router(q)
    llm_r = llm_router.classify(q)
    rule_m  = '' if rule  == expected else '!'
    emb_m   = '' if emb   == expected else '!'
    llm_m   = '' if llm_r == expected else '!'
    print(f'{q:<52} {expected:<9} {rule+rule_m:<9} {emb+emb_m:<9} {llm_r+llm_m}')
print()
print('! = misclassified vs. expected label.')
print('LLM router handles ambiguous hybrid cases the rule router misses.')


---
## 3. Text-to-SQL: From Vanilla to Self-Debugging

Given a user question + schema → produce valid, executable SQL.

Four quality levels, each building on the previous:

```
Vanilla (zero-shot)  →  Few-shot  →  Chain-of-Thought  →  Execution Refinement
```

Each step costs more tokens and latency but catches more failure modes.


In [ ]:
# ── Schema DDL (what the LLM sees) ──────────────────────────────────────────

SCHEMA_DDL = """\
-- regions(id, name, launch_year)
-- customers(id, name, email, region, signup_date, tier)   tier IN ('free','pro','enterprise')
-- products(id, name, category, unit_price)               category IN ('Software','Hardware','Service')
-- orders(id, customer_id, order_date, status, total_amount) status IN ('shipped','pending','refunded','cancelled')
-- order_items(id, order_id, product_id, quantity, unit_price)
--
-- Foreign keys: orders.customer_id -> customers.id
--              order_items.order_id -> orders.id
--              order_items.product_id -> products.id
"""

# ── Mock Text-to-SQL LLM ─────────────────────────────────────────────────────
class MockTextToSQL:
    """
    Pattern-matches questions to pre-written SQL templates.
    Deliberately injects a bug in some queries to demo the self-debug loop.
    Replace generate() with a real LLM call in Section 7.
    """

    TEMPLATES = [
        ('how many.*refund',
         "SELECT COUNT(*) AS refund_count FROM orders WHERE status='refunded'"),
        ('total revenue|revenue this',
         "SELECT ROUND(SUM(total_amount),2) AS total_revenue FROM orders WHERE status='shipped'"),
        ('top.*customer',
         "SELECT c.name, ROUND(SUM(o.total_amount),2) AS revenue "
         "FROM customers c JOIN orders o ON o.customer_id=c.id "
         "WHERE o.status='shipped' GROUP BY c.id,c.name ORDER BY revenue DESC LIMIT 5"),
        ('average order',
         "SELECT ROUND(AVG(total_amount),2) AS avg_order FROM orders WHERE status='shipped'"),
        ('customer.*region|region.*customer',
         "SELECT region, COUNT(*) AS customer_count FROM customers GROUP BY region ORDER BY customer_count DESC"),
        ('inactive|haven.*order.*90',
         "SELECT c.name, MAX(o.order_date) AS last_order FROM customers c "
         "LEFT JOIN orders o ON o.customer_id=c.id "
         "GROUP BY c.id,c.name HAVING last_order < date('now','-90 days') OR last_order IS NULL"),
        ('pro.*tier|tier.*pro',
         "SELECT COUNT(*) AS pro_count FROM customers WHERE tier='pro'"),
        ('order.*last month|last month.*order',
         "SELECT COUNT(*) AS orders_last_month FROM orders "
         "WHERE order_date >= date('now','-30 days')"),
    ]

    def generate(self, question, schema=None, examples=None, with_cot=False, inject_bug=False):
        q = question.lower()
        for pattern, sql in self.TEMPLATES:
            if re.search(pattern, q):
                if inject_bug:
                    sql = sql.replace('total_amount', 'amount')  # column doesn't exist
                return sql
        return 'SELECT COUNT(*) AS count FROM orders'

    def fix_sql(self, broken_sql, error_msg):
        """Simulate self-debug: replace the bad column name."""
        fixed = broken_sql.replace('amount', 'total_amount')
        return fixed


text2sql = MockTextToSQL()

# ── Vanilla zero-shot demo ────────────────────────────────────────────────────
print('=== Vanilla (zero-shot) Text-to-SQL ===\n')
zero_shot_prompt = ROUTER_PROMPT  # just schema + question, no examples

demo_questions = [
    'How many refunded orders do we have?',
    'What is the total revenue from shipped orders?',
    'Show me the top 5 customers by revenue.',
]
for q in demo_questions:
    sql = text2sql.generate(q)
    result = qry(sql)
    print(f'Q: {q}')
    print(f'  SQL: {sql}')
    print(f'  Result: {result}')
    print()


In [ ]:
# ── Few-shot Text-to-SQL: retrieval-augmented prompt construction ────────────
# Pick examples that are semantically similar to the incoming question.

FEW_SHOT_EXAMPLES = [
    {'question': 'How many orders do we have in total?',
     'sql': "SELECT COUNT(*) AS total_orders FROM orders"},
    {'question': 'What is the average order value for shipped orders?',
     'sql': "SELECT ROUND(AVG(total_amount),2) AS avg_order FROM orders WHERE status='shipped'"},
    {'question': 'Which customers signed up this year?',
     'sql': "SELECT name, signup_date FROM customers WHERE signup_date >= date('now','start of year')"},
    {'question': 'How many products are in each category?',
     'sql': "SELECT category, COUNT(*) AS count FROM products GROUP BY category"},
    {'question': 'What regions did we expand into after 2020?',
     'sql': "SELECT name, launch_year FROM regions WHERE launch_year > 2020"},
    {'question': 'Show me all pending orders.',
     'sql': "SELECT * FROM orders WHERE status='pending'"},
]

example_embeds = embedder.encode(
    [e['question'] for e in FEW_SHOT_EXAMPLES],
    convert_to_tensor=True, show_progress_bar=False)

def retrieve_examples(question, top_k=3):
    """Retrieve the most similar few-shot examples for this question."""
    q_emb  = embedder.encode(question, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, example_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(FEW_SHOT_EXAMPLES[i], float(scores[i])) for i in idx]

def build_few_shot_prompt(question, schema, top_k=3):
    examples = retrieve_examples(question, top_k)
    parts = [f'Schema:\n{schema}\n']
    parts.append('Examples:')
    for ex, score in examples:
        parts.append(f'  Q: {ex["question"]}')
        parts.append(f'  SQL: {ex["sql"]}\n')
    parts.append(f'Question: {question}\nSQL:')
    return '\n'.join(parts)


q = 'What is the average order value for shipped orders?'
prompt = build_few_shot_prompt(q, SCHEMA_DDL)
print('=== Few-shot prompt for: ===\n', q, '\n')
print(prompt)
print()
examples_used = retrieve_examples(q)
print('Examples retrieved by semantic similarity:')
for ex, score in examples_used:
    print(f'  score={score:.3f}  "{ex["question"]}')
print()
print('Tip: the examples teach the model your conventions (aliases, JOIN style)')
print('far more efficiently than instructions can.')


In [ ]:
# ── Chain-of-Thought (CoT) Text-to-SQL ──────────────────────────────────────
# Show reasoning steps before the SQL query.
# Helps the LLM avoid mistakes on multi-step questions.

class MockCoTTextToSQL(MockTextToSQL):
    COT_PLANS = {
        'inactive|haven.*order.*90': [
            '1. We need customers from the `customers` table.',
            '2. LEFT JOIN to `orders` to include customers with zero orders.',
            '3. Group by customer, take MAX(order_date).',
            "4. HAVING last_order < 90 days ago OR IS NULL (never ordered).",
        ],
        'top.*customer': [
            '1. Join `customers` with `orders` on customer_id.',
            "2. Filter orders WHERE status='shipped'.",
            '3. GROUP BY customer, SUM total_amount.',
            '4. ORDER BY revenue DESC, LIMIT 5.',
        ],
        'how many.*refund': [
            '1. Query the `orders` table.',
            "2. Filter WHERE status='refunded'.",
            '3. COUNT(*).',
        ],
    }

    def generate_with_cot(self, question):
        q = question.lower()
        plan = []
        for pattern, steps in self.COT_PLANS.items():
            if re.search(pattern, q):
                plan = steps
                break
        sql = self.generate(question)
        return plan, sql


cot_sql = MockCoTTextToSQL()

print('=== Chain-of-Thought Text-to-SQL ===\n')
cot_demos = [
    "Show me customers who haven't ordered in 90 days.",
    'Show me the top 5 customers by revenue.',
]
for q in cot_demos:
    plan, sql = cot_sql.generate_with_cot(q)
    result = qry(sql)
    print(f'Q: {q}')
    if plan:
        print('Reasoning:')
        for step in plan:
            print(f'  {step}')
    print(f'SQL: {sql}')
    print(f'Result ({len(result)} rows): {result[:3]}')
    print()

print('CoT works because multi-join queries have multiple decisions.')
print('Getting any one wrong breaks the whole query.')
print('CoT isolates each decision so the model can check its own work.')


In [ ]:
# ── Execution Refinement: self-debugging loop ────────────────────────────────
# Generate SQL -> try to run it -> catch errors -> feed back to LLM -> retry

def execute_with_refinement(question, max_attempts=3, inject_bug=False):
    """
    Text-to-SQL with a self-debugging retry loop.
    inject_bug=True deliberately introduces a bad column name to demo recovery.
    """
    sql      = text2sql.generate(question, inject_bug=inject_bug)
    attempts = []

    for attempt in range(1, max_attempts + 1):
        try:
            result = qry(sql)
            attempts.append({'attempt': attempt, 'sql': sql, 'status': 'OK',
                             'error': None, 'result': result})
            return True, attempts
        except Exception as e:
            err = str(e)
            attempts.append({'attempt': attempt, 'sql': sql, 'status': 'ERROR',
                             'error': err, 'result': None})
            if attempt < max_attempts:
                sql = text2sql.fix_sql(sql, err)   # LLM self-debug call

    return False, attempts


print('=== Execution Refinement (self-debugging loop) ===\n')

# Demo 1: query that works first time
print('-- Query 1: clean generation --')
ok, attempts = execute_with_refinement('What is the total revenue?')
for a in attempts:
    print(f'  Attempt {a["attempt"]}: {a["status"]}')
    if a['status'] == 'OK':
        print(f'    Result: {a["result"]}')

print()
# Demo 2: query with injected bug — watch the loop catch and fix it
print('-- Query 2: bad column name injected (amount vs total_amount) --')
ok, attempts = execute_with_refinement('What is the total revenue?', inject_bug=True)
for a in attempts:
    print(f'  Attempt {a["attempt"]}: {a["status"]}')
    print(f'    SQL: {a["sql"][:80]}')
    if a['status'] == 'ERROR':
        print(f'    Error: {a["error"]}')
        print(f'    -> Feeding error back to LLM for correction...')
    else:
        print(f'    Result: {a["result"]}')
print()
print('Without this loop: a single typo = a failed query and a frustrated user.')
print('With this loop: the system silently recovers from most syntax errors.')


---
## 4. Schema Linking — RAG Inside Your RAG

On a 200-table database, dumping the full schema into the LLM prompt will:
- Blow your context window
- Confuse the model (three tables named `status`)
- Cost a fortune per query

Schema linking retrieves only the relevant tables first — using vector search on
table descriptions. Yes, that's RAG to build the input for Text-to-SQL.

```
user question  →  embed  →  find similar table descriptions
               →  include only those tables in the SQL prompt
```


In [ ]:
# Each table gets a rich description: what it contains, column semantics, sample values.
# Embed these descriptions, then retrieve top-K for each incoming question.

TABLE_DESCRIPTIONS = [
    {'table': 'regions',
     'desc': 'Stores geographic regions where TechCo operates. '
             'Columns: id, name (e.g. North America, Europe, APAC), launch_year. '
             'Use when: filtering by geography, checking expansion dates.'},
    {'table': 'customers',
     'desc': 'All registered customers. '
             'Columns: id, name, email, region, signup_date, tier (free/pro/enterprise). '
             'Use when: customer counts, tier analysis, signup trends, geographic distribution.'},
    {'table': 'products',
     'desc': 'Product catalog. '
             'Columns: id, name, category (Software/Hardware/Service), unit_price. '
             'Use when: product performance, category revenue, price analysis.'},
    {'table': 'orders',
     'desc': 'Customer orders (header record). '
             'Columns: id, customer_id, order_date, status (shipped/pending/refunded/cancelled), total_amount. '
             'Use when: revenue totals, order counts, refund rates, time-series analysis.'},
    {'table': 'order_items',
     'desc': 'Line items within each order. '
             'Columns: id, order_id, product_id, quantity, unit_price. '
             'Use when: per-product revenue, quantity analysis, basket size.'},
]

table_embeds = embedder.encode(
    [t['desc'] for t in TABLE_DESCRIPTIONS],
    convert_to_tensor=True, show_progress_bar=False)

def schema_link(question, top_k=3):
    """Return the top-K most relevant table descriptions for this question."""
    q_emb  = embedder.encode(question, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, table_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(TABLE_DESCRIPTIONS[i], float(scores[i])) for i in idx]

def build_linked_schema(question, top_k=3):
    relevant = schema_link(question, top_k)
    lines    = []
    for t, score in relevant:
        lines.append(f"-- {t['table']}: {t['desc']}")
    return '\n'.join(lines)


schema_link_demos = [
    ('What is the total revenue from shipped orders?',         'Should find: orders'),
    ('How many customers are on the pro tier in APAC?',        'Should find: customers, regions'),
    ('Which products generate the most revenue by category?',  'Should find: products, order_items, orders'),
]

print('=== Schema Linking ===\n')
print('Full schema has 5 tables. Schema linker picks the relevant ones.\n')
for q, note in schema_link_demos:
    linked = schema_link(q, top_k=3)
    print(f'Q: {q}')
    print(f'   ({note})')
    for t, score in linked:
        print(f'   [{t["table"]}] score={score:.3f}')
    print()


---
## 5. The Full Hybrid RAG Pipeline

Wiring everything together:

```
question
    │
    ▼
  Router ──► sql ───► Schema Linking → Few-Shot Examples → Text-to-SQL → Execute (+ refinement)
    │
    ├──► vector ──► Embed + retrieve text chunks
    │
    └──► hybrid ──► both paths run in parallel
         │
         ▼
    Synthesis LLM
         │
         ▼
    Unified answer
```

The synthesis LLM sees hard numbers (from SQL) and soft context (from text).
It never has to guess facts it can look up.


In [ ]:
class MockSynthesisLLM:
    """
    Combines SQL result + text context into a human-readable answer.
    Replace with a real LLM call in production.
    """

    def synthesize(self, question, sql_result=None, doc_context=None, sql_query=None):
        parts = []

        if sql_result is not None and len(sql_result) > 0:
            row = sql_result[0] if len(sql_result) == 1 else sql_result
            # Format row as key: value pairs
            if isinstance(row, dict):
                data_str = ', '.join(f"{k}={v}" for k, v in row.items())
            else:
                data_str = str(row[:5])  # first 5 rows
            parts.append(f'[DB] {data_str}')

        if doc_context:
            parts.append(f'[Doc] {doc_context[:200]}')

        if not parts:
            return 'I could not find relevant information to answer this question.'

        return ' | '.join(parts)


synthesis = MockSynthesisLLM()


def hybrid_rag(question, verbose=True):
    """
    Full hybrid RAG pipeline:
      1. Route the question
      2a. SQL path: schema link → few-shot prompt → generate SQL → execute + refine
      2b. Vector path: embed query → retrieve text chunks
      3. Synthesize
    """
    steps  = {'question': question}

    # ── Step 1: Route ────────────────────────────────────────────────────────
    route  = llm_router.classify(question)
    steps['route'] = route
    if verbose:
        print(f'[Route] {route.upper()}')

    sql_result  = None
    doc_context = None
    sql_query   = None

    # ── Step 2a: SQL path ────────────────────────────────────────────────────
    if route in ('sql', 'hybrid'):
        linked_schema = build_linked_schema(question, top_k=3)
        examples      = retrieve_examples(question, top_k=2)
        sql_query     = text2sql.generate(question)
        steps['sql'] = sql_query

        ok, attempts = execute_with_refinement(question)
        if ok:
            sql_result = attempts[-1]['result']
            steps['sql_attempts'] = len(attempts)
        else:
            steps['sql_error'] = attempts[-1]['error']

        if verbose:
            print(f'[SQL] {sql_query}')
            print(f'[SQL result] {sql_result}')

    # ── Step 2b: Vector path ─────────────────────────────────────────────────
    if route in ('vector', 'hybrid'):
        results = vector_retrieve(question, top_k=2)
        doc_context = ' '.join(doc['text'][:200] for doc, _ in results)
        steps['docs'] = [(doc['id'], round(score, 3)) for doc, score in results]

        if verbose:
            top_doc, top_score = results[0]
            print(f'[Vector] [{top_doc["id"]}] score={top_score:.3f}')

    # ── Step 3: Synthesize ───────────────────────────────────────────────────
    answer = synthesis.synthesize(question, sql_result, doc_context, sql_query)
    steps['answer'] = answer

    if verbose:
        print(f'[Answer] {answer}')

    return steps


print('hybrid_rag() ready.')


In [ ]:
# ── Demo: run questions from the blog exercises through the pipeline ──────────

PIPELINE_DEMOS = [
    ("What's our return policy for electronics?",       'vector'),
    ("How many refunded orders do we have?",             'sql'),
    ("Why did returns spike in March, and by how much?", 'hybrid'),
    ("Show me customers who haven't ordered in 90 days.",'sql'),
    ("What is the average order value?",                 'sql'),
    ("What is our shipping policy?",                     'vector'),
]

for q, expected_route in PIPELINE_DEMOS:
    print(f'=== Q: {q} ===')
    result = hybrid_rag(q, verbose=True)
    match  = 'OK' if result['route'] == expected_route else f'MISMATCH (expected {expected_route})'
    print(f'[Route check] {match}')
    print()


In [ ]:
# ── Pipeline architecture visualization ──────────────────────────────────────

fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16)
ax.set_ylim(0, 8)
ax.axis('off')

def box(ax, x, y, w, h, text, fc='#1565C0', tc='white', fs=9):
    ax.add_patch(mpatches.FancyBboxPatch(
        (x-w/2, y-h/2), w, h,
        boxstyle='round,pad=0.12', facecolor=fc, edgecolor='#CCCCCC', lw=1.5))
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fs, color=tc, fontweight='bold', multialignment='center')

def arr(ax, x1, y1, x2, y2, color='#333333', label='', lpos='top'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        dy = 0.18 if lpos == 'top' else -0.18
        ax.text(mx, my+dy, label, fontsize=8, color='#555555', ha='center')

# Question input
box(ax, 2, 6.5, 2.5, 0.75, 'User\nQuestion', '#37474F')

# Router
box(ax, 5.5, 6.5, 2.5, 0.75, 'Query\nRouter', '#F57F17', 'black')
arr(ax, 3.25, 6.5, 4.25, 6.5)

# SQL path
box(ax, 5.5, 4.8, 2.2, 0.65, 'Schema\nLinking', '#1565C0')
box(ax, 8.2, 4.8, 2.2, 0.65, 'Few-Shot\nExamples', '#1565C0')
box(ax, 10.9, 4.8, 2.2, 0.65, 'Text-to-SQL\nGenerate', '#1565C0')
box(ax, 13.5, 4.8, 2.2, 0.65, 'Execute +\nRefine Loop', '#0D47A1')

arr(ax, 5.5, 6.12, 5.5, 5.12, '#1565C0', 'sql/hybrid')
arr(ax, 6.6, 4.8, 7.1, 4.8, '#1565C0')
arr(ax, 9.3, 4.8, 9.8, 4.8, '#1565C0')
arr(ax, 12.0, 4.8, 12.4, 4.8, '#1565C0')

# Vector path
box(ax, 5.5, 2.5, 2.2, 0.65, 'Embed\nQuery', '#2E7D32')
box(ax, 8.2, 2.5, 2.2, 0.65, 'Retrieve\nText Chunks', '#2E7D32')

arr(ax, 5.5, 6.12, 5.5, 2.82, '#2E7D32', 'vector/hybrid')
arr(ax, 6.6, 2.5, 7.1, 2.5, '#2E7D32')

# Synthesis
box(ax, 10.9, 1.1, 2.5, 0.75, 'Synthesis\nLLM', '#6A1B9A')
arr(ax, 13.5, 4.47, 13.5, 1.5, '#0D47A1')
ax.annotate('', xy=(11.5, 1.47), xytext=(11.5, 2.17),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.8))

# Output
box(ax, 13.8, 1.1, 2.5, 0.75, 'Unified\nAnswer', '#37474F')
arr(ax, 12.15, 1.1, 12.55, 1.1, '#6A1B9A')

# Labels
ax.text(9.5, 5.6, 'SQL Path  (facts, numbers, aggregates)', ha='center',
        fontsize=9, color='#1565C0', fontweight='bold')
ax.text(7.0, 1.9, 'Vector Path  (policies, explanations, context)', ha='center',
        fontsize=9, color='#2E7D32', fontweight='bold')

ax.set_title('Hybrid RAG Pipeline Architecture', fontsize=14, fontweight='bold', y=0.97)
show_plot()


In [ ]:
# ── Evaluation: vector-only vs. SQL-only vs. hybrid ─────────────────────────

EVAL_SET = [
    # Questions only answerable from the DB
    {'q': 'How many refunded orders do we have?',
     'key': 'refund_count', 'needs': 'sql', 'label': 'Refund count (DB)'},
    {'q': 'What is the total revenue from shipped orders?',
     'key': 'total_revenue', 'needs': 'sql', 'label': 'Total revenue (DB)'},
    {'q': "Show me customers who haven't ordered in 90 days.",
     'key': 'last_order',   'needs': 'sql', 'label': 'Inactive customers (DB)'},
    {'q': 'How many pro-tier customers do we have?',
     'key': 'pro_count',    'needs': 'sql', 'label': 'Pro tier count (DB)'},
    # Questions only answerable from docs
    {'q': 'What is our return policy for electronics?',
     'key': '30 days',      'needs': 'vector', 'label': 'Return policy (doc)'},
    {'q': 'How much does expedited shipping cost?',
     'key': '$24.99',       'needs': 'vector', 'label': 'Shipping cost (doc)'},
    {'q': 'What do enterprise customers get that pro customers do not?',
     'key': 'account manager', 'needs': 'vector', 'label': 'Tier benefits (doc)'},
    # Hybrid questions
    {'q': 'How many orders were refunded and what does our refund policy say?',
     'key': 'refund',       'needs': 'hybrid', 'label': 'Refunds + policy (hybrid)'},
]


def score_system(system_fn, questions):
    scores = []
    for item in questions:
        answer = system_fn(item['q'])
        hit    = item['key'].lower() in answer.lower()
        scores.append(1.0 if hit else 0.0)
    return scores


def vector_only(q):
    results = vector_retrieve(q, top_k=2)
    return ' '.join(d['text'] for d, _ in results)


def sql_only(q):
    sql = text2sql.generate(q)
    try:
        result = qry(sql)
        return str(result)
    except Exception:
        return ''


def hybrid_only(q):
    r = hybrid_rag(q, verbose=False)
    return r.get('answer', '')


vec_scores    = score_system(vector_only, EVAL_SET)
sql_scores    = score_system(sql_only,    EVAL_SET)
hybrid_scores = score_system(hybrid_only, EVAL_SET)

print(f'{"Label":<30} {"Needs":<9} {"Vector":<9} {"SQL":<9} {"Hybrid"}')
print('-' * 68)
for item, vs, ss, hs in zip(EVAL_SET, vec_scores, sql_scores, hybrid_scores):
    v = 'YES' if vs else 'NO'
    s = 'YES' if ss else 'NO'
    h = 'YES' if hs else 'NO'
    print(f'{item["label"]:<30} {item["needs"]:<9} {v:<9} {s:<9} {h}')
print()
print(f'Vector-only : {np.mean(vec_scores):.0%} ({int(sum(vec_scores))}/{len(vec_scores)})')
print(f'SQL-only    : {np.mean(sql_scores):.0%} ({int(sum(sql_scores))}/{len(sql_scores)})')
print(f'Hybrid      : {np.mean(hybrid_scores):.0%} ({int(sum(hybrid_scores))}/{len(hybrid_scores)})')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-question
labels = [e['label'][:25] for e in EVAL_SET]
x = np.arange(len(EVAL_SET))
w = 0.26
axes[0].bar(x - w, vec_scores,    w, color='#E53935', alpha=0.85, label='Vector-only')
axes[0].bar(x,     sql_scores,    w, color='#1565C0', alpha=0.85, label='SQL-only')
axes[0].bar(x + w, hybrid_scores, w, color='#2E7D32', alpha=0.85, label='Hybrid')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
axes[0].set_ylim(-0.05, 1.3)
axes[0].set_ylabel('Answer found (1=yes, 0=no)')
axes[0].set_title('Per-Question Results', fontweight='bold')
axes[0].legend(fontsize=9)

# Shade DB vs doc vs hybrid zones
n_sql = sum(1 for e in EVAL_SET if e['needs'] == 'sql')
n_vec = sum(1 for e in EVAL_SET if e['needs'] == 'vector')
axes[0].axvspan(-0.5, n_sql-0.5,            alpha=0.04, color='blue')
axes[0].axvspan(n_sql-0.5, n_sql+n_vec-0.5, alpha=0.04, color='red')
axes[0].axvspan(n_sql+n_vec-0.5, len(EVAL_SET)-0.5, alpha=0.04, color='green')
axes[0].text(n_sql/2-0.5, 1.22,       'DB questions', ha='center', fontsize=8, color='#0D47A1')
axes[0].text(n_sql+n_vec/2-0.5, 1.22, 'Doc questions', ha='center', fontsize=8, color='#B71C1C')
axes[0].text(n_sql+n_vec, 1.22,       'Hybrid', ha='center', fontsize=8, color='#1B5E20')

# Right: overall
systems = ['Vector\nOnly', 'SQL\nOnly', 'Hybrid']
avgs    = [np.mean(vec_scores), np.mean(sql_scores), np.mean(hybrid_scores)]
clrs    = ['#E53935', '#1565C0', '#2E7D32']
bars    = axes[1].bar(systems, avgs, color=clrs, alpha=0.85, width=0.5)
axes[1].set_ylim(0, 1.2)
axes[1].set_ylabel('Average accuracy')
axes[1].set_title('Overall Accuracy Comparison', fontweight='bold')
for bar, val in zip(bars, avgs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val+0.03,
                 f'{val:.0%}', ha='center', fontsize=13, fontweight='bold')

show_plot()
print('Neither vector-only nor SQL-only can answer all question types.')
print('Hybrid covers both — the router decides which path to take.')


---
## 6. Production Concerns

A checklist of things that will bite you before Monday's demo:

1. **Read-only access** — the LLM will eventually generate `DROP TABLE` if you let it
2. **Query timeouts + row limits** — `SELECT *` on a billion-row table is a bad time
3. **PII filtering** — the LLM doesn't know `users.ssn` should never appear in results
4. **Caching** — same question twice shouldn't hit the DB twice
5. **Observability** — log every (question, SQL, result) triple


In [ ]:
# ── Safety 1: Read-only connection ──────────────────────────────────────────
# In SQLite: set query_only PRAGMA.
# In PostgreSQL: CREATE USER rag_readonly; GRANT SELECT ON ALL TABLES ...

safe_conn = sqlite3.connect(':memory:')
safe_conn.row_factory = sqlite3.Row
# Copy data from the main db
for line in conn.iterdump():
    try:
        safe_conn.execute(line)
    except Exception:
        pass
safe_conn.execute('PRAGMA query_only = ON')  # enforce read-only at DB level
safe_conn.commit()

print('=== Read-only enforcement ===\n')
try:
    safe_conn.execute("INSERT INTO customers (name) VALUES ('Hacker')")
    print('DANGER: write succeeded — not safe!')
except Exception as e:
    print(f'Write attempt blocked: {e}')
print('SELECT still works:', safe_conn.execute('SELECT COUNT(*) FROM customers').fetchone()[0], 'customers')

print()
# ── Safety 2: LIMIT injection ─────────────────────────────────────────────────
print('=== LIMIT injection ===\n')

def safe_sql(sql, row_limit=1000):
    """Inject LIMIT if missing; block obviously dangerous statements."""
    sql_upper = sql.strip().upper()
    if not sql_upper.startswith('SELECT'):
        raise ValueError(f'Only SELECT allowed, got: {sql_upper[:20]}')
    if 'LIMIT' not in sql_upper:
        sql = sql.rstrip('; \n') + f' LIMIT {row_limit}'
    return sql

dangerous = 'DROP TABLE customers'
no_limit  = 'SELECT * FROM orders'
has_limit = 'SELECT * FROM orders LIMIT 5'

for stmt in [dangerous, no_limit, has_limit]:
    try:
        safe = safe_sql(stmt)
        print(f'IN : {stmt}')
        print(f'OUT: {safe}')
    except ValueError as e:
        print(f'IN : {stmt}')
        print(f'BLOCKED: {e}')
    print()

# ── Safety 3: PII column filter ───────────────────────────────────────────────
print('=== PII column filtering ===\n')
PII_COLUMNS = {'email', 'ssn', 'phone', 'credit_card', 'dob'}

def strip_pii(rows):
    """Remove PII columns from query results before sending to synthesis LLM."""
    return [{k: v for k, v in row.items() if k.lower() not in PII_COLUMNS}
            for row in rows]

raw = qry('SELECT id, name, email, region, tier FROM customers LIMIT 3')
safe_rows = strip_pii(raw)
print('Before PII filter:', raw[0])
print('After  PII filter:', safe_rows[0])
print('email removed before synthesis LLM sees the result.')

print()
# ── Safety 4: Embedding-based cache ──────────────────────────────────────────
print('=== Embedding-based query cache ===\n')

class SemanticCache:
    """Cache (question, sql, result) triples; retrieve by embedding similarity."""

    def __init__(self, threshold=0.92):
        self.threshold  = threshold
        self.entries    = []   # list of {emb, question, sql, result}
        self.hits, self.misses = 0, 0

    def lookup(self, question):
        if not self.entries:
            self.misses += 1
            return None
        q_emb  = embedder.encode(question, convert_to_tensor=True, show_progress_bar=False)
        embs   = np.vstack([e['emb'] for e in self.entries])
        from sentence_transformers import util as st_util
        scores = st_util.cos_sim(q_emb, embs)[0].cpu().numpy()
        best   = int(np.argmax(scores))
        if scores[best] >= self.threshold:
            self.hits += 1
            return self.entries[best]
        self.misses += 1
        return None

    def store(self, question, sql, result):
        emb = embedder.encode(question, show_progress_bar=False)
        self.entries.append({'emb': emb, 'question': question, 'sql': sql, 'result': result})


cache = SemanticCache(threshold=0.90)
q1 = 'How many refunded orders do we have?'
q2 = 'What is the count of refunded orders?'  # paraphrase
q3 = 'What is the total shipped revenue?'      # different question

# First call: miss
print(f'Q1: "{q1}"')
hit = cache.lookup(q1)
print(f'  Cache: {"HIT" if hit else "MISS"}')
if not hit:
    sql = text2sql.generate(q1)
    res = qry(sql)
    cache.store(q1, sql, res)
    print(f'  Stored. Result: {res}')

# Second call (paraphrase): should hit
print(f'Q2: "{q2}" (paraphrase)')
hit = cache.lookup(q2)
print(f'  Cache: {"HIT" if hit else "MISS"}')
if hit:
    print(f'  Returned cached result: {hit["result"]}')

# Third call: different question, miss
print(f'Q3: "{q3}" (different)')
hit = cache.lookup(q3)
print(f'  Cache: {"HIT" if hit else "MISS"}')
print(f'\nCache stats: {cache.hits} hits, {cache.misses} misses')


---
## 7. Claude API — Real Text-to-SQL

Replace `MockTextToSQL` with a real Claude call. The schema, few-shot examples,
and execution refinement loop all stay the same — only `generate()` changes.

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`


In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

SQL_GENERATION_PROMPT = """\
You are a SQL expert. Given a database schema and a user question, write a valid SQLite query.

Rules:
- Return ONLY the SQL query, no explanation
- Use table aliases for readability
- Always add LIMIT 1000 unless the question asks for a specific count
- Use date('now', '-N days') for relative date filters in SQLite

Schema:
{schema}

Examples:
{examples}

Question: {question}
SQL:"""

ROUTER_CLASSIFY_PROMPT = """\
Classify this question into exactly one category and respond with one word.

Categories:
- sql    : counting, aggregating, filtering records, specific numbers from a database
- vector : policies, explanations, qualitative information from documents
- hybrid : needs both database facts AND document context

Question: {question}
Category:"""

if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_router(question):
        resp = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=10,
            messages=[{'role': 'user', 'content':
                       ROUTER_CLASSIFY_PROMPT.format(question=question)}]
        )
        label = resp.content[0].text.strip().lower()
        return label if label in ('sql', 'vector', 'hybrid') else 'vector'

    def claude_text2sql(question, schema, examples_str):
        prompt = SQL_GENERATION_PROMPT.format(
            schema=schema, examples=examples_str, question=question)
        resp = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=400,
            messages=[{'role': 'user', 'content': prompt}]
        )
        sql = resp.content[0].text.strip()
        # Strip markdown fences if present
        sql = re.sub(r'^```sql\s*', '', sql, flags=re.MULTILINE)
        sql = re.sub(r'^```\s*$', '', sql, flags=re.MULTILINE)
        return sql.strip()

    def claude_hybrid_rag(question):
        route = claude_router(question)
        print(f'[Claude Router] {route.upper()}')

        sql_result  = None
        doc_context = None

        if route in ('sql', 'hybrid'):
            linked_schema = build_linked_schema(question, top_k=3)
            examples      = retrieve_examples(question, top_k=2)
            examples_str  = '\n'.join(
                f'Q: {e["question"]}\nSQL: {e["sql"]}' for e, _ in examples)
            sql = claude_text2sql(question, linked_schema, examples_str)
            print(f'[Claude SQL] {sql}')
            for attempt in range(3):
                try:
                    sql_result = qry(safe_sql(sql))
                    print(f'[Execute] OK on attempt {attempt+1}: {sql_result[:3]}')
                    break
                except Exception as e:
                    err = str(e)
                    print(f'[Execute] Error attempt {attempt+1}: {err}')
                    fix_prompt = (
                        f'The following SQL raised an error:\n{sql}\nError: {err}\n'
                        f'Schema:\n{linked_schema}\nReturn only the corrected SQL.')
                    resp = client.messages.create(
                        model='claude-sonnet-4-6',
                        max_tokens=300,
                        messages=[{'role': 'user', 'content': fix_prompt}]
                    )
                    sql = resp.content[0].text.strip()

        if route in ('vector', 'hybrid'):
            results     = vector_retrieve(question, top_k=2)
            doc_context = ' '.join(d['text'] for d, _ in results)

        return {'route': route, 'sql_result': sql_result, 'doc_context': doc_context}


    claude_questions = [
        'What is the total revenue from shipped orders?',
        "Show me customers who haven't ordered in 90 days.",
        'What is our return policy for electronics?',
    ]
    print('=== Claude Hybrid RAG ===\n')
    for q in claude_questions:
        print(f'Q: {q}')
        result = claude_hybrid_rag(q)
        print()

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('Prompt templates above map to three components:')
    print('  ROUTER_CLASSIFY_PROMPT  -> claude_router()      uses claude-haiku-4-5-20251001')
    print('  SQL_GENERATION_PROMPT   -> claude_text2sql()    uses claude-sonnet-4-6')
    print('  (inline fix prompt)     -> self-debug loop      uses claude-sonnet-4-6')
    print()
    print('Model split rationale:')
    print('  Router is a simple YES/NO classification — Haiku is fast and costs ~10x less.')
    print('  SQL generation requires schema understanding — Sonnet is worth the extra token cost.')
    print()
    print('Key tip: Always end the SQL_GENERATION_PROMPT with "SQL:" (no newline) so the')
    print('model starts generating SQL directly without preamble text.')


---
## Key Takeaways

1. **Vector search ≠ data retrieval.** It retrieves *text about* your data.
   For counting, summing, or ranking, it will retrieve the right page
   and then the LLM will confidently invent a number. Use SQL.

2. **The router is the most important component.** Getting the tool right matters
   more than getting the retrieval perfect. A misrouted question cannot be fixed
   by a better embedding model.

3. **Start with a rule-based router, graduate to embedding-based.** Rule-based routers
   are nearly free. Once you have 50+ labelled examples, embedding-based routers
   handle paraphrasing better. Add LLM-based routing only if the other two fail.

4. **Few-shot examples beat zero-shot for SQL generation.** Even one relevant example
   teaches the model your table aliases, JOIN style, and naming conventions.
   Retrieve examples by semantic similarity to the incoming question.

5. **Schema linking is non-negotiable at scale.** On a 200-table schema, flooding
   the prompt is expensive and confusing. Embed table descriptions and retrieve
   the relevant tables first. That's RAG inside your RAG.

6. **The self-debugging loop is the biggest reliability improvement.** Most SQL
   failures are syntax errors or wrong column names — both fixable if you catch
   the error and send it back to the LLM. Without the loop, one typo = one
   failed query. With it, the system silently recovers.

7. **Read-only access is not optional.** Set it at the database level with
   a dedicated SELECT-only user. Inject `LIMIT` if missing. Strip PII before
   results reach the synthesis LLM. These are defense-in-depth, not paranoia.

8. **Log every (question, SQL, result) triple.** When a user asks why the bot said
   revenue was $4M when it's $40M, this log is the only way to trace what happened.

---

*Up next — Lesson 9.3: My documents are messy — tables, charts, and footnotes.*
*How do I RAG over PDFs that aren't just text?*
